In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from sklearn.cluster import KMeans\n",
    "from src.data.loader import load_raw_data\n",
    "from src.data.preprocessor import encode_features\n",
    "from src.models.clustering import train_kmeans, assign_personas, save_cluster_model\n",
    "from pathlib import Path\n",
    "\n",
    "# Load and preprocess\n",
    "df = load_raw_data(Path('data/raw/leads.csv'))\n",
    "df_processed, _ = encode_features(df)\n",
    "\n",
    "# Select features for clustering (excluding target-like columns)\n",
    "cluster_features = ['engagement_score', 'email_opens', 'website_visits', 'annual_revenue', 'days_since_contact']\n",
    "X_cluster = df_processed[cluster_features]\n",
    "\n",
    "# Elbow method to determine k\n",
    "inertias = []\n",
    "k_range = range(1, 11)\n",
    "for k in k_range:\n",
    "    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')\n",
    "    kmeans.fit(X_cluster)\n",
    "    inertias.append(kmeans.inertia_)\n",
    "\n",
    "plt.figure(figsize=(8,5))\n",
    "plt.plot(k_range, inertias, 'bo-')\n",
    "plt.xlabel('Number of clusters (k)')\n",
    "plt.ylabel('Inertia')\n",
    "plt.title('Elbow Method for Optimal k')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train KMeans with k=3\n",
    "kmeans = train_kmeans(X_cluster, n_clusters=3)\n",
    "save_cluster_model(kmeans, 'models/kmeans_model.pkl')\n",
    "\n",
    "# Assign personas\n",
    "df_with_personas = assign_personas(df, kmeans, cluster_features)\n",
    "print(df_with_personas[['lead_id', 'company_name', 'engagement_score', 'email_opens', 'persona']].head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize clusters (2D projection using first two features)\n",
    "plt.figure(figsize=(10,6))\n",
    "scatter = plt.scatter(X_cluster['engagement_score'], X_cluster['email_opens'], c=kmeans.labels_, cmap='viridis')\n",
    "plt.xlabel('Engagement Score')\n",
    "plt.ylabel('Email Opens')\n",
    "plt.title('KMeans Clusters (k=3)')\n",
    "plt.colorbar(scatter)\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}